## Overview

This notebook is intended to:
* Introduce core views and functions available in Awesense's Energy Data Models (EDM).
* Demonstrate various ways to fetch and work with data using SQL and Python.

For complete documentation on the available views and functions, please refer to the [edm_overview.ipynb](edm_overview.ipynb) notebook.

---

## Set up

In [1]:
import getpass
import urllib.parse

**Connection**

Enter the full EDM server address to connect to (e.g. sandbox-edm.awesense.com), and the login credentials provided by Awesense.
<span style='color:red'> **Please do NOT store the credentials in the notebook, nor share them with anyone.** </span>

In [2]:
edm_address = getpass.getpass(prompt='EDM server address: ')

print('\nEDM login information')
edm_name = getpass.getpass(prompt='Username: ')
edm_password = getpass.getpass(prompt='Password: ')
edm_password = urllib.parse.quote(edm_password)

%load_ext sql
%sql postgresql://$edm_name:$edm_password@$edm_address/edm
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = False

# Delete the credential variables for security purpose.
del edm_name, edm_password

EDM server address: ········

EDM login information
Username: ········
Password: ········


---

## Views

**Grid**

The term `grid` represents a portion of a utility's grid, and the term `grid element` represents the physical elements that comprise the grid.

The view named `grid` lists available grids in the database, and `grid_id` is the primary key of the grid.

In [3]:
%%sql

SELECT *
FROM grid;

grid_id,description,last_updated
awefice,awefice,2021-01-07 00:00:00+00:00


**Grid Element**

The `grid_element` view contains information about grid elements in a grid. Properties that are common to all grid elements, such as `phases`, are explicitly listed as columns of the view. And additional metadata that may differ between grid elements of different types (e.g. meter vs. transformers, etc.) are stored as JSONB in the `meta` column. 

The `grid_id` column from the `grid_element` view references the `grid_id` from the `grid` view. And the `grid_id` and `grid_element_id` columns are often used as input arguments to various functions, as demonstrated in the **Functions** section below and in the [tgi_tracing.ipynb](tgi_tracing.ipynb) notebook. 

In [4]:
%%sql

SELECT *
FROM grid_element
WHERE grid_id = 'awefice'
LIMIT 1;

grid_id,grid_element_id,type,customer_type,phases,is_underground,is_producer,is_consumer,is_switchable,switch_is_open,terminal1_cn,terminal2_cn,power_flow_direction,upstream_grid_element_id,geometry,meta
awefice,transformer_2,Transformer,transformer,ABC,False,True,False,False,None,con_1,con_3,both,None,0101000020E61000002B8A5759DBC65EC07AA4038A3FA24840,"{'name': 'tr_2', 'label': 'tr_2', 'latitude': 49.26756406, 'load_loss': 246932, 'longitude': -123.107138, 'mv_feeder': None, 'ownership': 'utility', 'make_model': None, 'rating_kva': 20000, 'enclosure_id': None, 'no_load_loss': 24775, 'voltage_level': 'HV/MV', 'commission_date': '1989-01-01', 'primary_voltage': 200000, 'maintenance_type': 'oil', 'manufacture_date': None, 'installation_type': 'station', 'secondary_voltage': 12500, 'hvmv_parent_element': None, 'is_operation_status': True, 'parent_transformer_id': None, 'winding_configuration': None}"


Below are different grid element types available in the `awefice` grid. 

A CSV file detailing all grid element types in the `Sandbox` server and their respective properties that the Awesense Platform recognizes can be found [here](https://sandbox.awesense.com/docs/_static/gis_metadata.csv).

In [5]:
%%sql

SELECT DISTINCT type as element_type
FROM grid_element
WHERE grid_id = 'awefice'
ORDER BY element_type;

element_type
ACLineSegment
Battery
Busbar
Cabinet
CircuitBreaker
Disconnector
EVCharger
Fuse
Jumper
Meter


A complete list of properties available in the `meta` column for a grid element can be returned using the built-in `json_each_text()` function. The following is an example of accessing each item in the `meta` column with the 'awefice' `grid_id` and the 'm_12' `grid_element_id`.

In [6]:
%%sql

SELECT ge.grid_element_id, 
        meta_data.key, 
        meta_data.value
FROM grid_element ge
JOIN json_each_text(ge.meta::json) AS meta_data
    ON true
WHERE grid_id = 'awefice'
    AND grid_element_id = 'm_12';

grid_element_id,key,value
m_12,name,None
m_12,label,m_12
m_12,sdp_id,None
m_12,address,None
m_12,interval,3600
m_12,latitude,49.2667165
m_12,estimated,None
m_12,longitude,-123.10284
m_12,lv_feeder,None
m_12,mv_feeder,None


Specific properties from the `meta` column can be fetched individually as well.

In [7]:
%%sql

SELECT meta ->> 'longitude' as longitude,
        meta ->> 'latitude' as latitude
FROM grid_element 
WHERE grid_id = 'awefice'
    AND grid_element_id = 'm_12';

longitude,latitude
-123.10284,49.2667165


**Grid Element Data Source**

The `grid_element_data_source` view represents the linking between time-series data and physical elements on the grid. The primary key of this view is `grid_element_data_source_id`, and is used as an input argument of the `ts_data_source_select()` function to retrieve time series data, as demonstrated in the **Functions** section below and in the [time_series_access.ipynb](time_series_access.ipynb) notebook.

In [8]:
%%sql

SELECT *
FROM grid_element_data_source
LIMIT 5;

grid_element_data_source_id,grid_id,grid_element_id,valid,phases,type,provider,metrics,friendly_id,direction
5143f413-ae1e-4f3e-afdf-320a0a287efb,awefice,m_11,"[2021-01-01 08:01:00+00:00, None)",ABC,CONSUMER,voltage_reads,"['kWh', 'V']",m_11,None
4629171a-418d-46d3-90fe-b28a357ebf7e,awefice,m_10,"[2021-01-01 08:01:00+00:00, None)",ABC,CONSUMER,voltage_reads,"['kWh', 'V']",m_10,None
66e8132c-d413-486b-82ca-e02fdf6c866a,awefice,m_9,"[2021-01-01 08:01:00+00:00, None)",ABC,CONSUMER,voltage_reads,"['kWh', 'V']",m_9,None
a0eadf9f-7858-4db9-9a38-0f43374c5763,awefice,m_8,"[2021-01-01 08:01:00+00:00, None)",ABC,CONSUMER,voltage_reads,"['kWh', 'V']",m_8,None
a93a4af1-5c51-4ef8-b9d4-355a92a3dad1,awefice,m_7,"[2021-01-01 08:01:00+00:00, None)",ABC,CONSUMER,voltage_reads,"['kWh', 'V']",m_7,None


The `metrics` column is of type text[] and lists the units of measurements for time-series that can be retrieved for a given grid element. Please note that it is possible to have multiple metrics associated with the same `grid_element_data_source_id`. The following is an example of pulling out multiple grid element data sources available for the 'awefice' `grid_id` and the 'm_12' `grid_element_id`.

In [9]:
%%sql

SELECT geds.grid_element_data_source_id, 
        geds.grid_id, 
        geds.grid_element_id, 
        metric_key 
FROM grid_element_data_source geds
JOIN UNNEST(geds.metrics::TEXT[]) AS metric_key
    ON true
WHERE grid_id = 'awefice'
    AND grid_element_id = 'm_12';    

grid_element_data_source_id,grid_id,grid_element_id,metric_key
96ecaf9c-4c66-419d-a711-9d9027ff5f28,awefice,m_12,kWh
96ecaf9c-4c66-419d-a711-9d9027ff5f28,awefice,m_12,V


The `provider` column represents the origin of the stored data, and may be used to properly identify the grid_element_data_source for the time series of interest.

In [10]:
%%sql

SELECT distinct provider
FROM grid_element_data_source;

provider
scada_reads
state_of_charge
voltage_reads


---

## Functions

**grid_get_downstream()**

The `grid_get_downstream()` function is one of tracing functions available in EDM, and is mentioned here to demonstrate a function with grid and grid element arguments. Its input arguments can be found by running the following query.

Please refer to the [tgi_tracing.ipynb](tgi_tracing.ipynb) notebook for examples of all tracing functions.


In [11]:
%%sql

SELECT function_args
FROM get_function_documentation('grid_get_downstream');

function_args
"grid_id text, grid_element_id text, include_self boolean DEFAULT true"


Below is an example with the 'awefice' `grid_id`, the 'm_10' `grid_element_id` and False to exclude the specified `grid_element_id` from the output list.

In [12]:
%%sql

SELECT *
FROM grid_get_downstream('awefice', 'm_10', False);

grid_id,grid_element_id,type,customer_type,phases,is_underground,is_producer,is_consumer,is_switchable,switch_is_open,terminal1_cn,terminal2_cn,power_flow_direction,upstream_grid_element_id,geometry,meta
awefice,evch_10,EVCharger,ev_charger,ABC,None,True,True,False,None,con_135_t2,VCN.evch_10,both,None,0101000020E61000000B45A2539AC65EC0DDB431982CA24840,"{'name': None, 'label': 'evch_10', 'sdp_id': None, 'latitude': 49.26698591, 'longitude': -123.103169, 'max_current': 60, 'max_voltage': 480, 'is_dc_charger': True, 'voltage_level': 'LV', 'charging_level': 'level_2', 'connector_type': 'NEMA 14-40', 'active_charging_power': 6700, 'active_generation_power': 6000}"
awefice,batt_10,Battery,battery,ABC,None,True,True,False,None,con_135_t2,VCN.batt_10,both,None,0101000020E61000000B45A2539AC65EC0DDB431982CA24840,"{'name': 'batt_10', 'label': 'batt_10', 'sdp_id': None, 'latitude': 49.26698591, 'longitude': -123.103169, 'lv_feeder': None, 'mv_feeder': None, 'ownership': 'private', 'make_model': 'LG CHEM RESU10H', 'enclosure_id': None, 'voltage_level': 'LV', 'capacity_usable': 9.6, 'commission_date': None, 'capacity_nominal': 9.8, 'manufacture_date': None, 'efficiency_round_trip': 97.96, 'max_peak_charge_power': 7.0, 'parent_transformer_id': 'transformer_36', 'max_peak_discharge_power': 7.0, 'max_continuous_charge_power': 5.0, 'max_continuous_discharge_power': 5.0}"
awefice,pv_10,Photovoltaic,photovoltaic,ABC,None,True,True,False,None,con_135_t2,VCN.pv_10,both,None,0101000020E61000000B45A2539AC65EC0DDB431982CA24840,"{'name': 'pv_10', 'label': 'pv_10', 'module': 'LG Neon R', 'sdp_id': None, 'ac_size': 3600, 'dc_size': 3800, 'racking': 'roof', 'inverter': 'SolarEdge HDWave', 'latitude': 49.26698591, 'longitude': -123.103169, 'tilt_angle': 42, 'azimuth_angle': 170, 'voltage_level': 'LV', 'dc_to_ac_ratio': '90%', 'commission_date': None, 'number_of_modules': 10, 'degradation_annual': 0.7, 'generation_capacity': 3600, 'degradation_25_years': 17.5}"


**ts_data_source_select()**

The `ts_data_source_select()` returns time series data for the given `id`, `metric_key`, and `timerange` input arguments.

Please note that the `id` argument refers to the `grid_element_data_source_id` from the `grid_element_data_source` table discussed in the above. And the `metric_key` is one of the metrics listed in the `metrics` column in the same table. Below is an example of retrieving kWh time series for the 'm_12' grid_element (whose grid_element_data_source_id is '96ecaf9c-4c66-419d-a711-9d9027ff5f28') for April 1 ~ 30, 2022 time range.

In [13]:
%%sql

SELECT timestamp, value
FROM ts_data_source_select('96ecaf9c-4c66-419d-a711-9d9027ff5f28', 'kWh', '[2022-04-01, 2022-04-30]'
                          )
LIMIT 5;

timestamp,value
2022-04-01 00:00:00+00:00,3.84
2022-04-01 01:00:00+00:00,3.591
2022-04-01 02:00:00+00:00,3.646
2022-04-01 03:00:00+00:00,3.821
2022-04-01 04:00:00+00:00,3.531


More generally, time series for multiple grid elements and metrics over the entire period can be retrieved by joining multiple tables as the following. Please note that the query output is ordered and limited for a demonstration purpose.

In [14]:
%%sql

SELECT geds.grid_element_id, 
        metric_key, 
        tdss.timestamp, 
        tdss.value
FROM grid_element ge
JOIN grid_element_data_source geds
    ON geds.grid_id = ge.grid_id
    AND geds.grid_element_id = ge.grid_element_id
JOIN UNNEST(geds.metrics::TEXT[]) metric_key
    ON true
JOIN ts_data_source_select(geds.grid_element_data_source_id, metric_key) tdss
    ON true
WHERE ge.grid_id = 'awefice'
    AND ge.type = 'Meter'
ORDER by 3
LIMIT 5;

grid_element_id,metric_key,timestamp,value
m_2,kWh,2021-01-01 08:00:00+00:00,2.034
m_2,V,2021-01-01 08:00:00+00:00,119.114
m_1,kWh,2021-01-01 08:00:00+00:00,1.827
m_1,V,2021-01-01 08:00:00+00:00,119.566
m_3,kWh,2021-01-01 08:00:00+00:00,1.815


---

## SQL & Python

**Data Retrieval**

Below are a few ways to save SQL output into a local Python variable.

Option 1

In [15]:
%%sql result <<

SELECT grid_element_id, type
FROM grid_element
WHERE grid_id = 'awefice';

Returning data to local variable result


Option 2

In [16]:
# Save the sql output as a variable in one line.
result = %sql SELECT grid_element_id, type FROM grid_element WHERE grid_id = 'awefice'

Option 3

In [17]:
# Save the sql output as a variable in multiple lines.
result = %sql SELECT grid_element_id, type \
                FROM grid_element \
                WHERE grid_id = 'awefice'

Then the following code can be run to convert the above SQL output `result` into a Python dataframe to work with.

In [18]:
df = result.DataFrame()

# Return the first three lines of the dataframe to take a peek.
df.head(3)

,grid_element_id,type
0,transformer_2,Transformer
1,transformer_6,Transformer
2,line_segment_3,ACLineSegment


**Variable Input**

A python variable can be integrated into SQL queries by wrapping the variable with `'{}'`.

In [19]:
grid_element_id = 'm_12'

In [20]:
%%sql

SELECT *
FROM grid_element_data_source
WHERE grid_element_id = '{grid_element_id}';

grid_element_data_source_id,grid_id,grid_element_id,valid,phases,type,provider,metrics,friendly_id,direction
96ecaf9c-4c66-419d-a711-9d9027ff5f28,awefice,m_12,"[2021-01-01 08:01:00+00:00, None)",ABC,CONSUMER,voltage_reads,"['kWh', 'V']",m_12,None
